## Distance From xQTL Credible Set to TSS

Final selected panels for the single-context credible-set distance figure. This notebook is self-contained: it loads the packaged figure-data RDS, builds the selected main and supplementary panels, and saves the publication files to `../figure_data/`. The data currently store five distance intervals, so these panels summarize interval-level TSS distance distributions.


#### Setup and load data


In [1]:
library(dplyr)
library(ggplot2)
library(scales)
library(tidyr)
library(grid)

figure_data_dir <- file.path("..", "figure_data")
dir.create(figure_data_dir, showWarnings = FALSE, recursive = TRUE)
figure_out_dir <- normalizePath(file.path(".", "figure_data"), mustWork = FALSE)
dir.create(figure_out_dir, showWarnings = FALSE, recursive = TRUE)

summary_candidates <- c(
  file.path(figure_data_dir, "Figure_1_single_context_cs_distance.rds"),
  file.path(figure_data_dir, "Figure_1_single_context_cs_distance.rds.bak")
)
summary_path <- summary_candidates[file.exists(summary_candidates)][1]
if (is.na(summary_path) || !nzchar(summary_path)) {
  stop("Could not find Figure_1_single_context_cs_distance.rds or .rds.bak in ", figure_data_dir)
}
summary_obj <- readRDS(summary_path)

distance_levels <- c("<10Kb", "10Kb~50Kb", "50Kb~100Kb", "100Kb~1Mb", ">1Mb")
distance_labels <- c("< 10 kb", "10-50 kb", "50-100 kb", "100 kb-1 Mb", ">= 1 Mb")

assign_tissue_group <- function(resource) {
  dplyr::case_when(
    grepl("STARNET|ROSMAP|PCC|DLPFC|MSBB|Knight.*eQTL.*brain|AC.*DeJager|AC.*CUIMC1|BM.*MSBB|PCC.*DeJager|PCC.*CUIMC1", resource) ~ "Bulk brain",
    grepl("Inh|Exc", resource) ~ "Neurons",
    grepl("Knight.*pQTL|DLPFC.*Klein", resource) ~ "Other",
    grepl("MIGA|MiGA", resource) ~ "MiGA",
    grepl("Metabrain", resource) ~ "MetaBrain",
    TRUE ~ "Glia"
  )
}

assign_modality_group <- function(resource) {
  dplyr::case_when(
    grepl("sQTL", resource) ~ "sQTL",
    grepl("gpQTL", resource) ~ "gpQTL",
    grepl("pQTL", resource) ~ "pQTL",
    grepl("monocyte|Mac|Ast|Exc|Inh|Mic|OPC|Oli", resource) ~ "Cell-type eQTL",
    TRUE ~ "Bulk eQTL"
  )
}

clean_resource_label <- function(x) {
  x %>%
    gsub("_", " ", ., fixed = TRUE) %>%
    gsub("DeJager", "CUIMC1", ., fixed = TRUE) %>%
    gsub("Kellis", "MIT", ., fixed = TRUE) %>%
    gsub("STARNET eQTL Mac", "STARNET Mac eQTL", .) %>%
    gsub("monocyte ROSMAP eQTL", "ROSMAP monocyte eQTL", .) %>%
    gsub("Knight (eQTL|pQTL) brain", "Knight brain \\1", .) %>%
    gsub("BM (\\d+) MSBB eQTL", "MSBB BM\\1 eQTL", .) %>%
    gsub("DLPFC Bennett pQTL", "Bennett DLPFC pQTL", .) %>%
    gsub("DLPFC Klein (gpQTL adjusted|gpQTL unadjusted)", "Klein DLPFC \\1", .) %>%
    gsub("(Ast|Oli|Mic|OPC|Exc|Inh|AC|PCC|DLPFC) CUIMC1 eQTL", "CUIMC1 \\1 eQTL", .) %>%
    gsub("(Ast|Oli|Mic|OPC|Exc|Inh) MIT eQTL", "MIT \\1 eQTL", .) %>%
    gsub("(Ast|Mic) (\\d+) MIT eQTL", "MIT \\1\\2 eQTL", .) %>%
    gsub("(Ast|Oli|Mic|OPC|Exc|Inh) mega eQTL", "mega \\1 eQTL", .)
}

save_plot_pair <- function(plot, filename_stub, width, height, dpi = 300) {
  pdf_path <- file.path(figure_out_dir, paste0(filename_stub, ".pdf"))
  ggsave(pdf_path, plot, width = width, height = height, device = cairo_pdf)
  message("Saved: ", pdf_path)
  invisible(pdf_path)
}


Using poppler version 26.02.0



#### Prepare plotting data


In [2]:
summary_distance_data <- summary_obj$data$singlecontext_top_loci_table_integrated_distance_filtered_category %>%
  mutate(
    distance_category = factor(as.character(distance_category), levels = distance_levels),
    display_resource = clean_resource_label(ifelse(is.na(display_resource), resource, display_resource)),
    tissue_group = factor(assign_tissue_group(resource),
                          levels = c("Bulk brain", "Neurons", "Glia", "Other", "MiGA", "MetaBrain")),
    modality_group = factor(assign_modality_group(resource),
                            levels = c("Cell-type eQTL", "Bulk eQTL", "sQTL", "pQTL", "gpQTL"))
  )

resource_order <- summary_distance_data %>%
  distinct(resource, display_resource, tissue_group, modality_group, total_count) %>%
  arrange(tissue_group, desc(total_count), display_resource) %>%
  mutate(y_global = -row_number())

summary_distance_data <- summary_distance_data %>%
  left_join(resource_order %>% select(resource, y_global), by = "resource")

context_labels <- resource_order %>% mutate(y_context = y_global)

modality_axis_labels <- c(
  "Cell-type eQTL" = "Cell-type eQTL", "Bulk eQTL" = "Bulk eQTL",
  "sQTL" = "sQTL", "pQTL" = "pQTL", "gpQTL" = "gpQTL"
)

modality_colors <- c(
  "Cell-type eQTL" = "#2F6FB0",
  "Bulk eQTL" = "#7AA6D9",
  "sQTL" = "#B54852",
  "pQTL" = "#D9896C",
  "gpQTL" = "#8B6FAE"
)



#### Main figure: proximal and distal summary


In [3]:
interval_metrics <- summary_distance_data %>%
  group_by(resource, display_resource, modality_group, tissue_group) %>%
  summarise(
    `<= 100 kb` = sum(frequency[distance_category %in% distance_levels[1:3]], na.rm = TRUE),
    `>= 1 Mb` = sum(frequency[distance_category == ">1Mb"], na.rm = TRUE),
    .groups = "drop"
  ) %>%
  pivot_longer(c(`<= 100 kb`, `>= 1 Mb`), names_to = "metric", values_to = "fraction") %>%
  mutate(metric = factor(metric, levels = c("<= 100 kb", ">= 1 Mb")))

p_main_cs_distance_proximal_distal_summary <- ggplot(
  interval_metrics,
  aes(x = modality_group, y = fraction, fill = modality_group, color = modality_group)
) +
  geom_boxplot(width = 0.58, alpha = 0.35, outlier.shape = NA, linewidth = 0.55) +
  geom_point(
    position = position_jitter(width = 0.12, height = 0, seed = 1),
    size = 1.8,
    alpha = 0.68
  ) +
  facet_wrap(~ metric, nrow = 1) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1),
                     expand = expansion(mult = c(0.01, 0.04))) +
  scale_x_discrete(labels = modality_axis_labels) +
  scale_fill_manual(values = modality_colors, guide = "none") +
  scale_color_manual(values = modality_colors, guide = "none") +
  labs(x = NULL, y = "Fraction of events") +
  theme_bw(base_size = 14) +
  theme(
    axis.text.x = element_text(size = 11, color = "black", angle = 25, hjust = 1),
    axis.text.y = element_text(size = 12, color = "black"),
    axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 8)),
    strip.text = element_text(size = 13, face = "bold"),
    strip.background = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    plot.margin = margin(12, 12, 6, 8)
  )



#### Supplementary figure: interval stacked context profile


In [4]:
summary_stacked_yfacet <- summary_distance_data %>%
  mutate(display_resource_ordered = factor(display_resource, levels = rev(resource_order$display_resource)))

p_supp_cs_distance_interval_stacked_yfacet <- ggplot(summary_stacked_yfacet) +
  geom_col(
    aes(x = frequency, y = display_resource_ordered, fill = distance_category),
    orientation = "y",
    position = position_stack(reverse = TRUE),
    width = 0.78,
    color = "grey35",
    linewidth = 0.12
  ) +
  facet_grid(tissue_group ~ ., scales = "free_y", space = "free_y") +
  scale_fill_manual(
    values = c(
      "<10Kb" = "#2F6FB0",
      "10Kb~50Kb" = "#7AA6D9",
      "50Kb~100Kb" = "#D9E6F2",
      "100Kb~1Mb" = "#E8C6BD",
      ">1Mb" = "#B54852"
    ),
    breaks = distance_levels,
    labels = distance_labels,
    name = "Distance to TSS",
    guide = guide_legend(nrow = 1, byrow = TRUE)
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), expand = expansion(mult = c(0, 0.01))) +
  coord_cartesian(xlim = c(0, 1), clip = "off") +
  scale_y_discrete(drop = TRUE) +
  labs(x = "Fraction of events", y = NULL) +
  theme_bw(base_size = 13) +
  theme(
    axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 8)),
    axis.text.x = element_text(size = 11.5, color = "black"),
    axis.text.y = element_text(size = 10.5, color = "black", face = "bold"),
    strip.text.y.right = element_text(size = 12, face = "bold", angle = 270),
    strip.background = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    legend.position = "top",
    legend.title = element_text(size = 11, face = "bold"),
    legend.text = element_text(size = 10),
    legend.margin = margin(0, 0, 0, 0),
    legend.box.margin = margin(0, 0, 4, 0),
    plot.margin = margin(14, 18, 6, 6)
  )



#### Supplementary figure: proximal/intermediate/distal tail profile


In [5]:
tail_context_order <- summary_distance_data %>%
  group_by(resource, display_resource, tissue_group, modality_group) %>%
  summarise(
    proximal = sum(frequency[distance_category %in% distance_levels[1:2]], na.rm = TRUE),
    intermediate = sum(frequency[distance_category %in% distance_levels[3:4]], na.rm = TRUE),
    distal_tail = sum(frequency[distance_category == ">1Mb"], na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(tissue_group, desc(distal_tail), desc(proximal), display_resource) %>%
  mutate(y_tail = -row_number())

tail_profile <- tail_context_order %>%
  select(resource, display_resource, tissue_group, modality_group, y_tail, proximal, intermediate, distal_tail) %>%
  pivot_longer(c(proximal, intermediate, distal_tail),
               names_to = "distance_region", values_to = "fraction") %>%
  mutate(
    distance_region = factor(
      distance_region,
      levels = c("proximal", "intermediate", "distal_tail"),
      labels = c("< 50 kb", "50 kb-1 Mb", ">= 1 Mb")
    )
  )

p_supp_cs_distance_tail_profile <- ggplot(tail_profile) +
  geom_point(aes(x = fraction, y = y_tail, color = distance_region), size = 2.2) +
  geom_line(aes(x = fraction, y = y_tail, group = resource), color = "grey74", linewidth = 0.35) +
  facet_grid(tissue_group ~ ., scales = "free_y", space = "free_y") +
  scale_color_manual(
    values = c("< 50 kb" = "#2F6FB0", "50 kb-1 Mb" = "#BDBDBD", ">= 1 Mb" = "#B54852"),
    name = NULL,
    guide = guide_legend(nrow = 1, byrow = TRUE)
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1),
                     expand = expansion(mult = c(0.01, 0.03))) +
  scale_y_continuous(
    breaks = tail_context_order$y_tail,
    labels = tail_context_order$display_resource,
    expand = expansion(add = c(0.4, 0.4))
  ) +
  labs(x = "Fraction of events", y = NULL) +
  theme_bw(base_size = 13) +
  theme(
    axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 8)),
    axis.text.x = element_text(size = 11.5, color = "black"),
    axis.text.y = element_text(size = 10.5, color = "black", face = "bold"),
    strip.text.y.right = element_text(size = 12, face = "bold", angle = 270),
    strip.background = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    legend.position = "top",
    legend.text = element_text(size = 10.5),
    legend.margin = margin(0, 0, 0, 0),
    legend.box.margin = margin(0, 0, 4, 0),
    plot.margin = margin(14, 18, 6, 6)
  )



#### Save selected deliverables


## Main Figure Panel 1d -- Compact Distance Distribution


In [ ]:
distance_modality_distribution <- summary_distance_data %>%
  group_by(modality_group, distance_category) %>%
  summarise(fraction = mean(frequency, na.rm = TRUE), .groups = "drop") %>%
  group_by(modality_group) %>%
  mutate(fraction = fraction / sum(fraction, na.rm = TRUE)) %>%
  ungroup()

p_main_cs_distance_distribution_by_modality <- ggplot(
  distance_modality_distribution,
  aes(x = modality_group, y = fraction, fill = distance_category)
) +
  geom_col(width = 0.68, color = "grey30", linewidth = 0.18) +
  scale_fill_manual(
    values = c("<10Kb" = "#EFF3FF", "10Kb~50Kb" = "#BDD7E7",
               "50Kb~100Kb" = "#6BAED6", "100Kb~1Mb" = "#F4A261",
               ">1Mb" = "#C73E4A"),
    breaks = distance_levels, labels = distance_labels, name = "Distance to TSS",
    guide = guide_legend(nrow = 2, byrow = TRUE)
  ) +
  scale_y_continuous(labels = percent_format(accuracy = 1), expand = expansion(mult = c(0, 0.02))) +
  scale_x_discrete(labels = modality_axis_labels) +
  labs(x = NULL, y = "Mean fraction") +
  coord_cartesian(ylim = c(0, 1), clip = "off") +
  theme_bw(base_size = 15) +
  theme(
    axis.text.x = element_text(size = 12, color = "black", angle = 25, hjust = 1),
    axis.text.y = element_text(size = 12, color = "black"),
    axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 7)),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    legend.position = "top",
    legend.title = element_text(size = 11, face = "bold"),
    legend.text = element_text(size = 10),
    legend.key.width = unit(1.1, "lines"),
    legend.margin = margin(0, 0, 0, 0),
    legend.box.margin = margin(0, 0, 2, 0),
    plot.margin = margin(8, 12, 4, 8)
  )


In [6]:
selected_output_files <- tibble::tribble(
  ~role, ~figure, ~pdf,
  "main", "proximal_distal_summary",
  file.path(figure_out_dir, "Main_Figure1_single_context_cs_distance_proximal_distal_summary.pdf"),
  "main", "distance_distribution_by_modality",
  file.path(figure_out_dir, "Main_Figure1_single_context_cs_distance_distribution_by_modality.pdf"),
  "supplementary", "interval_stacked_yfacet",
  file.path(figure_out_dir, "Supplementary_Figure1_single_context_cs_distance_interval_stacked_yfacet.pdf"),
  "supplementary", "tail_profile",
  file.path(figure_out_dir, "Supplementary_Figure1_single_context_cs_distance_tail_profile.pdf")
)

save_plot_pair(
  p_main_cs_distance_proximal_distal_summary,
  "Main_Figure1_single_context_cs_distance_proximal_distal_summary",
  width = 4.8, height = 4.3
)
save_plot_pair(
  p_main_cs_distance_distribution_by_modality,
  "Main_Figure1_single_context_cs_distance_distribution_by_modality",
  width = 4.8, height = 4.4
)
save_plot_pair(
  p_supp_cs_distance_interval_stacked_yfacet,
  "Supplementary_Figure1_single_context_cs_distance_interval_stacked_yfacet",
  width = 9.8, height = 17
)
save_plot_pair(
  p_supp_cs_distance_tail_profile,
  "Supplementary_Figure1_single_context_cs_distance_tail_profile",
  width = 9.8, height = 17
)

selected_output_files


role,figure,pdf,png
<chr>,<chr>,<chr>,<chr>
main,proximal_distal_summary,../figure_data/Main_Figure1_single_context_cs_distance_proximal_distal_summary.pdf,../figure_data/Main_Figure1_single_context_cs_distance_proximal_distal_summary.png
supplementary,interval_stacked_yfacet,../figure_data/Supplementary_Figure1_single_context_cs_distance_interval_stacked_yfacet.pdf,../figure_data/Supplementary_Figure1_single_context_cs_distance_interval_stacked_yfacet.png
supplementary,tail_profile,../figure_data/Supplementary_Figure1_single_context_cs_distance_tail_profile.pdf,../figure_data/Supplementary_Figure1_single_context_cs_distance_tail_profile.png


#### Main figure preview

Compares the fraction of events close to TSS (`<= 100 kb`) and in the distal tail (`>= 1 Mb`) across QTL types.

![](../figure_data/Main_Figure1_single_context_cs_distance_proximal_distal_summary.png)


#### Supplementary figure preview: interval stacked context profile

![](../figure_data/Supplementary_Figure1_single_context_cs_distance_interval_stacked_yfacet.png)


#### Supplementary figure preview: proximal/intermediate/distal tail profile

![](../figure_data/Supplementary_Figure1_single_context_cs_distance_tail_profile.png)
